In [1]:
# ============================================================
# STEP 2B — Load Cleaned CSVs into MySQL
# File: step2b_load_mysql.py
# Libraries used: pandas, mysql-connector-python
#
# Install (run once in terminal):
#   pip install mysql-connector-python openpyxl pandas
#
# Run AFTER:
#   step1_clean_data.py          (creates the 3 CSV files)
#   step2a_create_schema.sql     (run in MySQL Workbench first!)
# ============================================================

In [2]:
pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import mysql.connector
from mysql.connector import Error

In [4]:
# ── CONFIG — edit these to match your MySQL setup ─────────────
MYSQL_HOST     = "127.0.0.1"
MYSQL_PORT     = 3306
MYSQL_USER     = "root"
MYSQL_PASSWORD = "Anjumbano12@#$%1"    # <-- CHANGE THIS
MYSQL_DATABASE = "netflix_db"
 

In [5]:
# ── CONNECT ──────────────────────────────────────────────────
print("Connecting to MySQL Workbench ...")
try:
    conn = mysql.connector.connect(
        host     = MYSQL_HOST,
        port     = MYSQL_PORT,
        user     = MYSQL_USER,
        password = MYSQL_PASSWORD,
        database = MYSQL_DATABASE
    )
    cursor = conn.cursor()
    cursor.execute("SELECT VERSION()")
    ver = cursor.fetchone()
    print(f"  Connected! MySQL version: {ver[0]}")
except Error as e:
    print(f"  ERROR: {e}")
    print("  Check your MYSQL_PASSWORD and that MySQL Workbench server is running.")
    exit(1)

Connecting to MySQL Workbench ...
  Connected! MySQL version: 8.0.39


In [6]:
def bulk_insert(df, table_name):
    cols = ", ".join(df.columns)
    placeholders = ", ".join(["%s"] * len(df.columns))

    sql = f"INSERT IGNORE INTO {table_name} ({cols}) VALUES ({placeholders})"

    data = []
    for row in df.itertuples(index=False):
        clean_row = []
        for v in row:
            if pd.isna(v):
                clean_row.append(None)
            else:
                clean_row.append(v)
        data.append(tuple(clean_row))

    cursor.executemany(sql, data) # insert one row
    conn.commit()
    return len(data)

In [7]:
# df_t["date_added"] = df_t["date_added"].dt.strftime("%Y-%m-%d")
df_t.head()


NameError: name 'df_t' is not defined

In [ ]:
# ── LOAD: netflix_titles ──────────────────────────────────────
print("\nLoading netflix_titles ...")
df = pd.read_csv("cleaned_netflix.csv", parse_dates=["date_added"])
# Select only the columns that exist in the table
cols_titles = ["show_id","type","title","director","cast","country",
               "date_added","year_added","month_added","month_name",
               "release_year","rating","audience_group","duration",
               "duration_minutes","duration_seasons","genres","description",
               "content_age_at_add"]
df_t = df[cols_titles].copy()
# Format date for MySQL
df_t["date_added"] = df_t["date_added"].dt.strftime("%Y-%m-%d")
n = bulk_insert(df_t,"netflix_titles")
print(f"  Inserted {n:,} rows into netflix_titles")



Loading netflix_titles ...
  Inserted 7,682 rows into netflix_titles


In [ ]:
# ── LOAD: netflix_genres ──────────────────────────────────────
print("\nLoading netflix_genres ...")
df_g = pd.read_csv("genres_exploded.csv")
cols_genres = ["show_id","type","title","release_year","year_added",
               "rating","audience_group","genre"]
df_g = df_g[cols_genres].copy()
n = bulk_insert(df_g, "netflix_genres")
print(f"  Inserted {n:,} rows into netflix_genres")



Loading netflix_genres ...
  Inserted 16,859 rows into netflix_genres


In [ ]:
# ── LOAD: netflix_countries ───────────────────────────────────
print("\nLoading netflix_countries ...")
df_c = pd.read_csv("countries_exploded.csv")
cols_countries = ["show_id","type","title","release_year","year_added",
                  "rating","country_single"]
df_c = df_c[cols_countries].copy()
n = bulk_insert(df_c, "netflix_countries")
print(f"  Inserted {n:,} rows into netflix_countries")
 


Loading netflix_countries ...
  Inserted 8,954 rows into netflix_countries


In [ ]:
# ── VERIFY ───────────────────────────────────────────────────
print("\n=== VERIFICATION ===")
for table in ["netflix_titles", "netflix_genres", "netflix_countries"]:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"  {table:<25} : {count:,} rows") 
cursor.close()
conn.close()
print("\nAll data loaded! Now run step2c_create_views.sql in MySQL Workbench.")



=== VERIFICATION ===
  netflix_titles            : 7,682 rows
  netflix_genres            : 16,859 rows
  netflix_countries         : 8,954 rows

All data loaded! Now run step2c_create_views.sql in MySQL Workbench.
